In [ ]:
!pip install ultralytics --upgrade
!pip install Pillow --upgrade
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --force-reinstall

In [2]:
import os
import shutil
import random

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO,YOLOWorld


In [3]:
yaml_content = """
path: /kaggle/working/VisDrone
test: VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images
train: VisDrone2019-DET-train/VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/VisDrone2019-DET-val/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/kaggle/working/VisDrone.yaml', 'w') as f:
    f.write(yaml_content)

In [4]:
src_dir = Path('/kaggle/input/datasets/kushagrapandya/visdrone-dataset/')
dst_dir = Path('/kaggle/working/VisDrone')

In [5]:
def visdrone2yolo(src_dir, dst_dir, num_samples=None):
    
    
    def convert_box(size, box):
        dw = 1. / size[0]
        dh = 1. / size[1]
        return (box[0] + box[2] / 2) * dw, (box[1] + box[3] / 2) * dh, box[2] * dw, box[3] * dh

    (dst_dir / 'labels').mkdir(parents=True, exist_ok=True)
    all_files = list((src_dir / 'annotations').glob('*.txt'))
    
    if num_samples:
        all_files = random.sample(all_files, min(num_samples, len(all_files)))
    
    for f in tqdm(all_files, desc=f'Converting {src_dir}'):
        img_size = Image.open((src_dir / 'images' / f.name).with_suffix('.jpg')).size
        lines = []
        with open(f, 'r') as file:
            for row in [x.split(',') for x in file.read().strip().splitlines()]:
                if row[4] == '0':
                    continue
                cls = int(row[5]) - 1
                box = convert_box(img_size, tuple(map(int, row[:4])))
                lines.append(f"{cls} {' '.join(f'{x:.6f}' for x in box)}\n")
        with open(dst_dir / 'labels' / f.name, 'w') as fl:
            fl.writelines(lines)


datasets = ['VisDrone2019-DET-test-dev']

for d in datasets:
    visdrone2yolo(src_dir / d / d, dst_dir / d / d)

Converting /kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:34<00:00, 46.21it/s]


In [6]:

for d in datasets:
    src_imgs = src_dir / d / d / 'images'
    dst_imgs = dst_dir / d / d / 'images'
    
    if src_imgs.exists():
        if dst_imgs.is_symlink():
            dst_imgs.unlink()
        elif dst_imgs.is_dir():
            shutil.rmtree(dst_imgs)
            
    # recreate
    dst_imgs.mkdir(parents=True, exist_ok=True)
    imgs = list(src_imgs.glob('*.jpg'))
    for img in tqdm(imgs, desc=f'Copying: {d}'):
        shutil.copy(img, dst_imgs / img.name)
dir1 = dst_dir/ 'VisDrone2019-DET-train'/'VisDrone2019-DET-train'/ 'images'
dir1.mkdir(parents=True, exist_ok=True)
dir2 = dst_dir/ 'VisDrone2019-DET-val' / 'VisDrone2019-DET-val' / 'images'
dir2.mkdir(parents=True, exist_ok=True)
print("\n End")

Copying: VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:02<00:00, 663.56it/s]


 End


# ==========================================
# 2. EVALUATE THE FINE TUNED MODEL
# ==========================================

In [7]:
print("Evaluating Fine-Tuned Model...")

# Point to the specific 'best.pt' generated by your training command
# Based on your CLI command, it should be in runs/visdrone_exp1/weights/best.pt
ft_model = YOLOWorld("/kaggle/input/datasets/penglufly/yw-trained-weights/weights/best.pt")

# Run validation. You DO NOT need to set classes here; they are baked into your best.pt!
ft_metrics = ft_model.val(data="/kaggle/working/VisDrone.yaml", split="test", save=True)

print(f"Fine-Tuned mAP50: {ft_metrics.box.map50:.4f}")
print(f"Overall mAP@0.5:0.95: {ft_metrics.box.map:.4f}") # 'map' without numbers means the 50-95 average

for i, class_name in enumerate(ft_model.names.values()):
    print(f"Class '{class_name}' - mAP50: {ft_metrics.box.maps[i]:.4f}")


print(ft_metrics.save_dir)


Evaluating Fine-Tuned Model...
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-worldv2 summary (fused): 88 layers, 12,749,288 parameters, 0 gradients, 34.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2368.2±516.0 MB/s, size: 184.5 KB)
val: Scanning /kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 1.1Kit/s 1.4s0.0s
val: New cache created: /kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 4.3it/s 23.5s0.2s
                   all       1610      75102      0.426      0.334      0.299      0.168
            pedestrian       1197      21006        0.4      0.283      0.249     0.0952
                people        797       6376      0.469      0.123      0.128     0.0399
        

# ==========================================
# 2. EVALUATE THE ZERO-SHOT BASE MODEL
# ==========================================

In [8]:
print("\nEvaluating Zero-Shot Model...")

# Load the raw, untrained base model
zs_model = YOLOWorld("yolov8s-worldv2.pt")

# CRUCIAL: Inject your custom vocabulary into the blank model
zs_model.set_classes([
    "pedestrian", "people", "bicycle", "car", "van", 
    "truck", "tricycle", "awning-tricycle", "bus", "motor"
])

# Run validation on the exact same test set
zs_metrics = zs_model.val(data="/kaggle/working/VisDrone.yaml", split="test", save=True)

print(f"Zero-Shot mAP50: {zs_metrics.box.map50:.4f}")
print(f"Overall mAP@0.5:0.95: {zs_metrics.box.map:.4f}") # 'map' without numbers means the 50-95 average

for i, class_name in enumerate(zs_model.names.values()):
    print(f"Class '{class_name}' - mAP50: {zs_metrics.box.maps[i]:.4f}")

print(zs_metrics.save_dir)


Evaluating Zero-Shot Model...
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 652ms
Prepared 2 packages in 2.16s
Installed 2 packages in 2ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@577b3cfa75f072c20a3f02b44ffde111d93eb3be)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 174MiB/s]


Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-worldv2 summary: 236 layers, 164,026,601 parameters, 151,277,313 gradients, 34.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1875.9±552.3 MB/s, size: 161.5 KB)
val: Scanning /kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/labels.cache... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 519.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 4.9it/s 20.7s0.2s
                   all       1610      75102      0.287      0.109     0.0649     0.0373
            pedestrian       1197      21006      0.453     0.0533     0.0559     0.0224
                people        797       6376      0.131     0.0412     0.0138    0.00388
               bicycle        377       1302      0.171     0.0261     0.0194    0.00631
                   car       1530      28074      0.512 

In [ ]:
shutil.make_archive('/kaggle/working/ft_result', 'zip', '/kaggle/working/runs/detect/val-8')
shutil.make_archive('/kaggle/working/zs_result', 'zip', '/kaggle/working/runs/detect/val-9')

In [9]:
def draw_gt(image, label_path, classes):
    img = image.copy()
    h, w = img.shape[:2]
    with open(label_path) as f:
        for line in f:
            cls_id, cx, cy, bw, bh = map(float, line.strip().split())
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, classes[int(cls_id)], (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

In [12]:
def draw_pred(image, results):
    if not results:
        return image
    return cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

In [ ]:
import matplotlib.pyplot as plt
import cv2
img_path =  "/kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images/0000006_00159_d_0000001.jpg"
gt_boxes =  "/kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/labels/0000006_00159_d_0000001.txt"
img_path1 =  "/kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images/0000054_00786_d_0000001.jpg"
gt_boxes1 =  "/kaggle/working/VisDrone/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/labels/0000054_00786_d_0000001.txt"

original = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# The confidence value is measured by the test set in the evaluation, see the F1 score curve.
zs_results = zs_model.predict(img_path, conf=0.059)
ft_results = ft_model.predict(img_path, conf=0.186)

gt_img    = draw_gt(original, gt_boxes, zs_model.names)
zs_pred_img  = draw_pred(original, zs_results)
ft_pred_img  = draw_pred(original, ft_results)



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))  

images = [original, gt_img, zs_pred_img, ft_pred_img]
titles = ["Original", "Ground Truth", "zero Shot", "Fine Tune"]

for ax, img, title in zip(axes.flatten(), images, titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/comparison.png", dpi=150, bbox_inches="tight")
plt.show()
